In [ ]:
#0. MERFISH preprocessing steps
##Part A — Split large images into tiled blocks for downstream nucleus segmentation (CellProfiler)
##Part B — Stitch block masks, detect nucleus-containing cells, and match transcripts to cells.

In [ ]:
"""
Part B — Stitch block masks, detect nucleus-containing cells, and match transcripts to cells.

Requirements:
  - After Part A, run CellProfiler to produce mask files named like '<blockname>_mask.tif'
    in the dapi blocks directory (or masks_dir). Example: dapi_block_0_0_mask.tif
  - Edit CONFIG below to point to files and mask folder.
"""

import os
import numpy as np
import pandas as pd
import tifffile
from tqdm import tqdm

# ---------------- CONFIG (EDIT) ----------------
DATA_DIR = os.path.abspath(os.environ.get("SPATIAL_VELO_RAW_DATA_DIR", os.path.join("..", "data", "raw")))
BLOCKS_ROOT = os.path.abspath(os.environ.get("SPATIAL_VELO_OUTPUT_DIR", os.path.join("..", "results", "step0_blocks")))
BLOCKS_DIR = os.path.join(BLOCKS_ROOT, "dapi_blocks")   # folder containing dapi blocks and blocks_info.csv
MASKS_DIR = BLOCKS_DIR   # where CellProfiler wrote mask files; could be same as BLOCKS_DIR or e.g. BLOCKS_DIR/masks
BLOCKS_INFO_CSV = os.path.join(BLOCKS_DIR, "blocks_info.csv")
GLOBAL_MASK_OUT = os.path.join(BLOCKS_DIR, "global_nucleus_mask.tif")  # output stitched mask
GLOBAL_MASK_NPY = os.path.join(BLOCKS_DIR, "global_nucleus_mask.npy")
METADATA_CSV = os.path.join(DATA_DIR, "metadata_nuc.csv")   # cell bounding boxes (min_x, max_x, min_y, max_y in microns)
TRANSCRIPTS_CSV = os.path.join(DATA_DIR, "transcripts_with_CellComp.csv")
MATCHED_TX_OUT = os.path.join(BLOCKS_ROOT, "transcripts_with_CellComp_nuc.csv")
CHUNK_SIZE = 1000000   # number of transcripts to process per chunk (tune depending on memory)

#-----------below parameter should be adjusted according to MERFISH image metadata----------------
SCALE_UM_PER_PX = 0.325  #or set AFFINE_3x3 = np.array([...]) and set SCALE_UM_PER_PX = None
AFFINE_3x3 = None   # optional, if you have an affine transform [ [a,b,tx],[c,d,ty],[0,0,1] ]

# ------------------------------------------------

def load_blocks_info(blocks_info_csv):
    if not os.path.exists(blocks_info_csv):
        raise FileNotFoundError(f"blocks_info.csv not found: {blocks_info_csv}")
    return pd.read_csv(blocks_info_csv)

def assemble_block_masks(blocks_info_df, masks_dir, global_shape=None, mask_suffix="_mask.tif"):
    """
    Stitch per-block mask files into a single global boolean mask.
    blocks_info_df must have columns: file, y_start, x_start, y_end, x_end
    masks_dir must contain mask files named like file.replace('.tif', mask_suffix)
    """
    # If global shape not given, infer from max end coordinates
    if global_shape is None:
        H = blocks_info_df['y_end'].max()
        W = blocks_info_df['x_end'].max()
    else:
        H, W = global_shape
    global_mask = np.zeros((int(H), int(W)), dtype=bool)

    for idx, r in blocks_info_df.iterrows():
        block_file = r['file']
        mask_name = os.path.splitext(block_file)[0] + mask_suffix
        mask_path = os.path.join(masks_dir, mask_name)
        if not os.path.exists(mask_path):
            print(f"Warning: mask not found for block {block_file} -> expected {mask_name}. Skipping this block.")
            continue
        block_mask = tifffile.imread(mask_path)
        # ensure boolean
        block_mask = (block_mask.astype(np.uint8) > 0)
        y0, y1, x0, x1 = int(r['y_start']), int(r['y_end']), int(r['x_start']), int(r['x_end'])
        global_mask[y0:y1, x0:x1] = np.logical_or(global_mask[y0:y1, x0:x1], block_mask[:(y1-y0), :(x1-x0)])
    # save
    tifffile.imwrite(GLOBAL_MASK_OUT, (global_mask.astype(np.uint8) * 255).astype(np.uint8))
    np.save(GLOBAL_MASK_NPY, global_mask)
    print(f"Saved global mask TIFF: {GLOBAL_MASK_OUT} and numpy: {GLOBAL_MASK_NPY}")
    return global_mask

def micron_to_pixel(x_um, y_um, scale_or_affine):
    """Convert micron coords to pixel coords (int). Accepts scalar um_per_px or 3x3 affine."""
    if scale_or_affine is None:
        raise RuntimeError("No scale or affine provided for micron_to_pixel conversion.")
    if isinstance(scale_or_affine, (int, float)):
        sx = sy = float(scale_or_affine)
        x_px = x_um / sx
        y_px = y_um / sy
    else:
        mat = np.asarray(scale_or_affine, dtype=float)
        if mat.shape == (3,3):
            v = mat.dot(np.array([x_um, y_um, 1.0]))
            x_px, y_px = v[0], v[1]
        else:
            raise ValueError("scale_or_affine must be scalar um_per_px or a 3x3 affine matrix.")
    return int(round(x_px)), int(round(y_px))

def micron_box_to_pixel_box(row, scale_or_affine):
    x1, y1 = micron_to_pixel(row['min_x'], row['min_y'], scale_or_affine)
    x2, y2 = micron_to_pixel(row['max_x'], row['max_y'], scale_or_affine)
    x1, x2 = sorted([x1, x2])
    y1, y2 = sorted([y1, y2])
    return x1, x2, y1, y2

def check_cell_has_nucleus(metadata_df, global_mask, scale_or_affine, cell_id_col='cell_id'):
    """
    Add a 'has_nucleus' boolean column to metadata_df by checking intersection with global_mask.
    metadata_df must contain min_x/max_x/min_y/max_y (in microns).
    """
    has_nucleus = []
    H, W = global_mask.shape
    for idx, row in metadata_df.iterrows():
        x1, x2, y1, y2 = micron_box_to_pixel_box(row, scale_or_affine)
        # clamp
        x1 = max(0, min(W-1, x1)); x2 = max(0, min(W, x2))
        y1 = max(0, min(H-1, y1)); y2 = max(0, min(H, y2))
        if x2 <= x1 or y2 <= y1:
            has_nucleus.append(False)
            continue
        region = global_mask[y1:y2, x1:x2]
        has_nucleus.append(bool(region.any()))
    metadata_df['has_nucleus'] = has_nucleus
    return metadata_df

def match_transcripts_to_cells(transcripts_csv, metadata_csv, out_csv, chunk_size=CHUNK_SIZE):
    """
    Vectorized chunked matching of transcripts to cell bounding boxes.
    metadata must contain: cell id (one of common names) and min_x,max_x,min_y,max_y (microns).
    Assigns a 'cell' column to transcripts by bounding box containment.
    """
    transcripts_cols = pd.read_csv(transcripts_csv, nrows=0).columns.tolist()
    if 'global_x' not in transcripts_cols or 'global_y' not in transcripts_cols:
        raise RuntimeError("transcripts file must contain 'global_x' and 'global_y' columns.")

    metadata = pd.read_csv(metadata_csv)
    # guess bounding box column names
    def pick(cands):
        for c in cands:
            if c in metadata.columns:
                return c
        raise RuntimeError(f"Cannot find any of columns: {cands} in metadata.")
    minx_col = pick(['min_x','xmin','x_min'])
    maxx_col = pick(['max_x','xmax','x_max'])
    miny_col = pick(['min_y','ymin','y_min'])
    maxy_col = pick(['max_y','ymax','y_max'])
    cell_id_col = pick(['cell','cell_id','Unnamed: 0','id'])

    cells = metadata[[cell_id_col, minx_col, maxx_col, miny_col, maxy_col]].copy()
    cells.columns = ['cell_id','min_x','max_x','min_y','max_y']
    cells['cell_id'] = cells['cell_id'].astype(str)

    # convert cell bbox arrays to numpy float
    minx = cells['min_x'].to_numpy().astype(float)
    maxx = cells['max_x'].to_numpy().astype(float)
    miny = cells['min_y'].to_numpy().astype(float)
    maxy = cells['max_y'].to_numpy().astype(float)

    # process transcripts in chunks
    reader = pd.read_csv(transcripts_csv, chunksize=int(chunk_size))
    out_chunks = []
    for chunk in tqdm(reader, desc="Matching transcripts"):
        coords = chunk[['global_x','global_y']].to_numpy(dtype=float)
        # vectorized boolean matrix: (n_chunk x n_cells)
        in_x = (coords[:,0][:,None] >= minx[None,:]) & (coords[:,0][:,None] <= maxx[None,:])
        in_y = (coords[:,1][:,None] >= miny[None,:]) & (coords[:,1][:,None] <= maxy[None,:])
        mask = in_x & in_y
        matched_any = mask.any(axis=1)
        # pick first matching cell index (if multiple boxes overlap choose first)
        first_idx = np.argmax(mask, axis=1)
        assigned = np.array([cells['cell_id'].to_numpy()[idx] if matched_any[i] else np.nan
                             for i, idx in enumerate(first_idx)], dtype=object)
        chunk['cell'] = assigned
        out_chunks.append(chunk)
    matched = pd.concat(out_chunks, axis=0)
    matched = matched.dropna(subset=['cell'])
    matched.to_csv(out_csv, index=False)
    print(f"Saved matched transcripts to {out_csv} (assigned {len(matched)} rows)")
    return out_csv

# ---------------- Run example ----------------
if __name__ == "__main__":
    # 1) Load blocks_info.csv that Part A wrote
    blocks_info = pd.read_csv(BLOCKS_INFO_CSV)
    print(f"Loaded {len(blocks_info)} blocks from {BLOCKS_INFO_CSV}")
    # 2) Assemble masks into global mask
    global_mask = assemble_block_masks(blocks_info, MASKS_DIR, global_shape=None, mask_suffix="_mask.tif")

    # 3) Load metadata and annotate has_nucleus
    meta = pd.read_csv(METADATA_CSV)
    if AFFINE_3x3 is not None:
        scale_or_affine = AFFINE_3x3
    else:
        scale_or_affine = SCALE_UM_PER_PX
    meta_with_flag = check_cell_has_nucleus(meta, global_mask, scale_or_affine)
    out_meta = os.path.join(os.path.dirname(METADATA_CSV), "cell_metadata_with_has_nucleus.csv")
    meta_with_flag.to_csv(out_meta, index=False)
    print(f"Saved metadata with has_nucleus flag: {out_meta}")

    # 4) Match transcripts to cells (heavy step)
    matched_out = match_transcripts_to_cells(TRANSCRIPTS_CSV, METADATA_CSV, MATCHED_TX_OUT, chunk_size=CHUNK_SIZE)
    print("Done.")
